In [1]:
import pandas as pd
import numpy as np
import os
import logging
from WeightedJaccardIndex import compute_weighted_jaccard_indices
seeds = [42,0,1,2,3]
# seeds = [42]
for seed in seeds:
    GSEA_results = {}
    synthetic_datasets = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",
                              f"synthpop_{seed}",f"tvae_{seed}"]
    or_gsea = pd.read_csv(f'CRPR_PD/Seed_{seed}/GSEA_Origin.csv')
    or_w = or_gsea.set_index('Term')['FDR q-val'].to_dict()
    sig_terms = or_gsea[or_gsea["FDR q-val"]< 0.05]['Term'].values.tolist()
    or_sig_nes = or_gsea[or_gsea["FDR q-val"]< 0.05].set_index('Term')['FDR q-val'].to_dict()
    or_nes = or_gsea.set_index('Term')['NES'].to_dict()
    # GSEA_results['Origin'] = or_gsea
    syn_w = {}
    syn_nes = {}
    sign_nes = {}
    for syntheticdata in synthetic_datasets:
        try:
            syn_gsea =  pd.read_csv(f'CRPR_PD/Seed_{seed}/GSEA_{syntheticdata}.csv')
            syn_w[syntheticdata] = syn_gsea.set_index('Term')['FDR q-val'].to_dict()
            syn_nes[syntheticdata] = syn_gsea.set_index('Term')['NES'].to_dict()
            sign_nes[syntheticdata] = syn_gsea[syn_gsea['Term'].isin(sig_terms)].set_index('Term')['NES'].to_dict()
        except:
            continue
    
    logging.basicConfig(level=logging.INFO)
    
    df_Jaccard = compute_weighted_jaccard_indices(
        original_pathways=or_w,
        synth_pathways_dict=syn_w,
        n_permutations=1000,
        seed=42,
        verbose=0,  # 0/1/2
    )
    df_Jaccard
    df_Jaccard.to_csv(f"CRPR_PD/Seed_{seed}/WeightedJaccardIndex_{seed}.csv", index=False)